# 05 ・ 接成服務：跑起完整的系統

## 這一章要做什麼

前四章做出了四個模組，但它們現在只有 notebook 用得到。
這一章把它們接成一個真的網頁服務 —— **包含前端** —— 然後直接看到成品。

```
瀏覽器
  │
  ├── GET  /              前端網頁（這一章寫出來）
  └── GET  /api/...       後端 API（這一章寫出來）
             │
             └── movieapp/   ← 你前四章寫的東西
```

## 關鍵設計：薄殼

服務層**不放任何邏輯**。每一支 API 只做三件事：
讀取請求參數 → 呼叫 `movieapp` 的函式 → 把結果包成 JSON。

這就是為什麼前四章要用 `%%writefile` 把邏輯寫成 `.py` ——
同一份程式碼，notebook 用它、網頁服務也用它，改一次兩邊都生效。

## 前端沒有 build 步驟

整個前端就是**一個 `index.html`**，樣式和程式都內嵌在裡面，
沒有框架、沒有打包工具、沒有 `node_modules`、沒有 `dist`。

代價是它比不上 React 那種寫法好維護，
但換來的是：**你看得到它每一行在做什麼**，
而且它跟其他檔案一樣是這本 notebook 寫出來的。

---
## 0 ・ 開場

這一章需要前面四本的產出，全部都要在。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")
from movieapp.config import setup
setup(requires=["sources", "tmdb", "merge", "gemini"])

In [ ]:
from movieapp import config
config.doctor();

---
## 1 ・ 建立 Django 專案骨架

Django 專案的檔案要放在固定的位置才找得到，先把資料夾建出來。

```
server/
├── manage.py          指令進入點
├── config/            專案設定（設定、網址、WSGI）
├── cinema/            我們的 app（API 視圖）
└── static/            前端網頁
```

接著馬上寫出兩個 `__init__.py`。它們裡面只有一段說明文字，
但**不能不放** —— 沒有 `__init__.py`，Python 不會把資料夾當成套件，
`import config.settings` 就會失敗。

In [ ]:
from pathlib import Path

for folder in ["../server", "../server/config", "../server/cinema", "../server/static"]:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print("  ", Path(folder).resolve())

In [ ]:
%%writefile ../server/config/__init__.py
"""Django 的專案設定套件。

這個檔案是空的（只有這段說明），但不能不放 ——
沒有 __init__.py，Python 不會把資料夾當成套件，
`import config.settings` 就會失敗。

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
"""

In [ ]:
%%writefile ../server/cinema/__init__.py
"""影城 API 的 Django app。

同樣是為了讓資料夾成為可 import 的套件而存在。

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
"""

---
## 2 ・ 專案設定

四個檔案，都是 Django 的固定樣板：

| 檔案 | 做什麼 |
|---|---|
| `manage.py` | 指令進入點，`runserver` 從這裡進去 |
| `config/settings.py` | 設定。注意它有多短 —— 沒有資料庫、沒有登入、沒有後台 |
| `config/wsgi.py` | 正式部署時 gunicorn 會找的進入點 |
| `cinema/apps.py` | app 的註冊資訊 |

`manage.py` 裡有一行值得看：

```python
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
```

它讓 `server/` 底下的程式找得到上一層的 `movieapp` —— 
這就是服務能 import 你前四章寫的東西的原因。

In [ ]:
%%writefile ../server/manage.py
#!/usr/bin/env python
"""Django 管理指令進入點。

一般不用手動執行，05_接成服務.ipynb 會用 subprocess 幫你啟動：
    python manage.py runserver 8008 --noreload

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""
import os
import sys
from pathlib import Path

# 讓 server/ 底下的程式找得到上一層的 movieapp 套件
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))


def main():
    os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
    from django.core.management import execute_from_command_line

    execute_from_command_line(sys.argv)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile ../server/config/settings.py
"""Django 設定。

這個服務只做兩件事：把 movieapp 的功能開成 API、以及把前端網頁送出去。
沒有資料庫、沒有登入、沒有後台，所以 INSTALLED_APPS 幾乎是空的。

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent
STATIC_DIR = BASE_DIR / "static"

# 教學用途，正式部署要改成從環境變數讀取
SECRET_KEY = "django-insecure-jupyter-teaching-package"
DEBUG = True
ALLOWED_HOSTS = ["*"]

INSTALLED_APPS = [
    "cinema",
]

MIDDLEWARE = [
    "django.middleware.common.CommonMiddleware",
]

ROOT_URLCONF = "config.urls"
WSGI_APPLICATION = "config.wsgi.application"

# 沒有用到資料庫
DATABASES = {}

LANGUAGE_CODE = "zh-hant"
TIME_ZONE = "Asia/Taipei"
USE_I18N = True
USE_TZ = True

# 前端和 API 都由這個服務提供（同一個網域），
# 所以不需要 CORS 設定，前端也不需要任何 build 步驟。

In [ ]:
%%writefile ../server/config/wsgi.py
"""WSGI 進入點。正式部署時 gunicorn／uwsgi 會找這個檔案。

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
"""

import os

from django.core.wsgi import get_wsgi_application

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
application = get_wsgi_application()

In [ ]:
%%writefile ../server/cinema/apps.py
"""Django app 設定。

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
"""

from django.apps import AppConfig


class CinemaConfig(AppConfig):
    name = "cinema"

---
## 3 ・ 薄殼：API 視圖

重點來了。**寫的時候注意每一支函式有多短。**

`/api/movies/` 是前端唯一需要的資料端點，它整條流程只有一行 ——
`merge.catalog()`，就是你第 3 章寫的那個函式。

其餘的端點（`/api/showtimes/`、`/api/tmdb/search/` …）保留下來，
是為了讓你能單獨戳每一層，除錯時很有用。

In [ ]:
%%writefile ../server/cinema/views.py
"""API 視圖 —— 全部都是薄殼。

注意每一支的長度：真正的邏輯都在 movieapp/ 裡，
這裡只負責「把 HTTP 請求翻譯成函式呼叫，再把結果包成 JSON」。

這就是為什麼要把邏輯抽出去：同一份程式碼，
notebook 直接 import 來用，這個服務也 import 來用，
改一個地方兩邊同時生效。

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

import json
import time
from pathlib import Path

from django.http import FileResponse, Http404, JsonResponse

from movieapp import gemini, merge, sources, tmdb

STATIC_DIR = Path(__file__).resolve().parent.parent / "static"


def _json(payload, error=None, status_on_error=502):
    """統一的回應格式：有錯誤就帶上 error 欄位和非 200 狀態碼。"""
    if error:
        return JsonResponse(dict(payload, error=error), status=status_on_error)
    return JsonResponse(payload)


# --------------------------------------------------------------------------
# 影城（01）
# --------------------------------------------------------------------------
def showtimes(request):
    titles, error = sources.showtimes()
    return _json({"movies": titles}, error)


def miramar(request):
    titles, error = sources.miramar()
    return _json({"movies": titles}, error)


# --------------------------------------------------------------------------
# TMDB（02）
# --------------------------------------------------------------------------
def tmdb_search(request):
    query = request.GET.get("query", "").strip()
    if not query:
        return JsonResponse({"error": "缺少 query 參數", "results": []}, status=400)

    language = request.GET.get("language", "zh-TW")
    results, error = tmdb.search(query, language=language)
    if error:
        return _json({"results": []}, error)

    # 前端只取 results[0]，所以這裡先把最像的排到最前面。
    # 不這樣做的話，TMDB 自己的相關度排序常常會讓「藍色監獄」
    # 拿到一部完全無關的電影。
    best = tmdb.best_match(query, results)
    if best is not None:
        results = [best] + [m for m in results if m.get("id") != best.get("id")]
    else:
        results = []

    return JsonResponse({"results": results})


def tmdb_genres(request):
    language = request.GET.get("language", "zh-TW")
    names, error = tmdb.genres(language=language)
    if error:
        return _json({"genres": []}, error)
    # 前端要的是 [{id, name}, ...] 這種形狀
    return JsonResponse({"genres": [{"id": k, "name": v} for k, v in names.items()]})


# --------------------------------------------------------------------------
# 整合後的電影清單（03）
# --------------------------------------------------------------------------
def movies(request):
    """前端唯一需要的資料端點：兩家影城 -> TMDB 補資料 -> 跨來源去重。

    整條流程都在 merge.catalog() 裡，這裡一樣只是薄殼。
    某一家影城掛掉不會讓整份清單失敗，錯誤放在 errors 欄位一起回去。
    """
    catalog, genre_names, errors = merge.catalog()
    return JsonResponse({
        "movies": catalog,
        "genres": [{"id": k, "name": v} for k, v in genre_names.items()],
        "labels": merge.SOURCE_LABELS,
        "errors": errors,
    })


# --------------------------------------------------------------------------
# 聊天（04）
# --------------------------------------------------------------------------
def chat(request):
    if request.method != "POST":
        return JsonResponse({"error": "請使用 POST"}, status=405)

    try:
        body = json.loads(request.body)
    except (json.JSONDecodeError, UnicodeDecodeError):
        return JsonResponse({"error": "無效的請求格式"}, status=400)

    messages = body.get("messages") or []
    latest = ""
    for message in reversed(messages):
        if message.get("role") == "user":
            latest = str(message.get("content", "")).strip()
            break
    if not latest:
        return JsonResponse({"error": "沒有使用者訊息"}, status=400)

    reply, error = gemini.ask(
        latest,
        system=str(body.get("system", "")).strip() or None,
        session_id=str(body.get("session_id", "")).strip() or None,
    )
    return _json({"reply": reply}, error)


# --------------------------------------------------------------------------
# 伺服器時間（05）
# --------------------------------------------------------------------------
def server_time(request):
    """回傳伺服器目前時間（UNIX 毫秒），供前端校時使用。

    瀏覽器沒辦法直接用 NTP（那是 UDP 123），所以改用 HTTP 對時：
    前端記錄送出與收到的時間，扣掉來回延遲的一半，推算出時鐘誤差。
    """
    return JsonResponse({"now": int(time.time() * 1000)})


# --------------------------------------------------------------------------
# 前端網頁
# --------------------------------------------------------------------------
def index(request):
    """把 build 好的前端首頁送出去。"""
    page = STATIC_DIR / "index.html"
    if not page.exists():
        raise Http404("找不到前端網頁，請確認 server/static/index.html 存在")
    return FileResponse(page.open("rb"), content_type="text/html")

每一支三到五行。所有真正的工作都在 `movieapp` 裡。

兩個地方值得停下來看：

**`tmdb_search` 為什麼比較長？** 因為它多做了一件事：
把最佳結果排到第一筆。前端只取 `results[0]`，
不這樣做的話「藍色監獄」就會拿到一部無關的電影（第 2 章示範過這個陷阱）。

**`movies` 為什麼不怕影城掛掉？** 因為 `catalog()` 把錯誤收在 `errors` 裡回傳，
不是丟例外。一家掛掉，另一家的電影照樣送到前端，
畫面上只會多一行紅字說明 —— 這是第 1 章 `(data, error)` 那個決定的回報。

In [ ]:
%%writefile ../server/cinema/urls.py
"""API 的網址對照表。

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
"""

from django.urls import path

from . import views

urlpatterns = [
    path("movies/", views.movies, name="movies"),
    path("showtimes/", views.showtimes, name="showtimes"),
    path("miramar/", views.miramar, name="miramar"),
    path("tmdb/search/", views.tmdb_search, name="tmdb-search"),
    path("tmdb/genres/", views.tmdb_genres, name="tmdb-genres"),
    path("chat/", views.chat, name="chat"),
    path("time/", views.server_time, name="server-time"),
]

---
## 4 ・ 網址設定

同一個服務同時提供 API 和網頁：

| 網址 | 內容 |
|---|---|
| `/api/...` | 後端 API |
| `/` | 前端首頁 |

**兩者在同一個網域**，所以不會有跨來源（CORS）問題，
前端用相對路徑 `/api/movies/` 就打得到自己的後端。

注意這裡**沒有 `/assets/` 這一條** —— 因為前端只有一個檔案，
沒有額外的 JS 和 CSS 要送。

In [ ]:
%%writefile ../server/config/urls.py
"""網址設定。

同一個服務同時提供兩件東西：
    /api/...   後端 API
    /          前端網頁（單一 index.html，樣式和程式都內嵌在裡面）

兩者在同一個網域底下，所以瀏覽器不會有跨來源問題，
前端也不需要 build 步驟 —— 沒有 Node.js、沒有打包工具、沒有 dist 資料夾。

本檔案由 notebooks/05_接成服務.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

from django.urls import include, path

from cinema.views import index

urlpatterns = [
    path("api/", include("cinema.urls")),
    path("", index, name="index"),
]

---
## 5 ・ 前端

整個網頁就是下面這一格。它很長，但結構只有三段：

| 段落 | 內容 |
|---|---|
| `<style>` | 版面和配色 |
| `<body>` | 三欄骨架：待看清單／最近瀏覽・電影清單・AI 聊天 |
| `<script>` | 跟 `/api/` 要資料、畫出來、把問題轉給 `/api/chat/` |

**這支前端不重複實作任何邏輯。** 篩選和排序在瀏覽器做（因為要即時反應，
每按一次按鈕都跑一趟伺服器太慢），但抓片單、查 TMDB、挑最佳比對結果
全都在後端的 `movieapp/` 裡，前端只負責畫面。

### 為什麼是「兩段式」而不是一次要完

前端沒有打 `/api/movies/`，而是分成兩步：

```
GET /api/<影城>/         -> 一串片名        （快，一次就好）
GET /api/tmdb/search/    -> 每個片名一次     （慢，六十幾次）
```

`/api/movies/` 會把這兩步合在伺服器做完才回應，好處是乾淨，
代價是**整個畫面要空白等到最後一部片查完**。分成兩步之後，
片單一到就先畫出來，海報和評分再一批一批補上去 ——
總時間差不多，但使用者第一秒就看得到東西。

### 幾個可以跟前面章節對照的地方

- `enrichTitles()` 用一個大小 8 的池子同時查 TMDB。
  一部一部排隊要三十幾秒，全部一次射出去會撞到 429，
  這就是第 2 章 `tmdb.enrich(workers=8)` 在瀏覽器這一側的版本
- `mergedMovies()` 用 TMDB id 當共同身分證跨影城去重 ——
  和第 3 章的 `merge.movie_key()` 是同一條規則
- `buildSystemPrompt()` 把**目前畫面上顯示的**電影塞進 prompt，
  跟第 4 章的 `gemini.build_system_prompt()` 是同一個想法（最陽春的 RAG）
- 待看清單存在 `localStorage`，換一台電腦就沒了 ——
  真的要跨裝置就得有帳號和資料庫，那是另一堂課的範圍


In [ ]:
%%writefile ../server/static/index.html
<!doctype html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>電影推薦系統</title>
<style>
/* --------------------------------------------------------------------------
 * 版面與配色。
 *
 * 這段樣式和原本用 Vite build 出來的前端逐條相同 —— 同樣的選擇器、
 * 同樣的數值，只是從壓縮過的一行還原成看得懂的形式。
 * 換句話說：畫面長什麼樣沒有變，變的只有「誰把它畫出來」。
 *
 * 最後兩條（* 和 body）是全域重置。它放在最後而不是最前面，
 * 是照著原本 build 產物的順序 —— * 的優先權是 0，
 * 永遠贏不過上面那些 class 選擇器，所以放前放後結果一樣。
 * -------------------------------------------------------------------------- */
.page {
  max-width: none;
  margin: 0 auto;
  padding: 24px 16px;
  display: flex;
  gap: 24px;
  align-items: flex-start;
}
.page-left {
  width: 280px;
  flex-shrink: 0;
  display: flex;
  flex-direction: column;
  gap: 16px;
  align-self: flex-start;
  position: sticky;
  top: 24px;
  max-height: calc(100vh - 48px);
}
.page-left .watchlist-block, .page-left .recent-block {
  flex: 1 1 auto;
  min-height: 0;
  max-height: calc((100vh - 64px)/2);
}
.page-main {
  flex: 1;
  min-width: 0;
}
.page-title {
  text-align: center;
  font-size: 2rem;
  margin-bottom: 24px;
  color: #1a202c;
}
.category-filter {
  background: #fff;
  border: 1px solid #e2e8f0;
  border-radius: 12px;
  padding: 16px;
  margin-bottom: 20px;
  box-shadow: 0 1px 3px #00000014;
  position: sticky;
  top: 12px;
  z-index: 30;
}
.category-filter-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 12px;
  flex-wrap: wrap;
  margin-bottom: 12px;
}
.category-filter-title {
  font-size: 1.1rem;
  font-weight: 600;
  color: #1a202c;
}
.category-filter-hint {
  font-size: .85rem;
  color: #718096;
}
.category-buttons {
  display: flex;
  gap: 8px;
  flex-wrap: wrap;
  justify-content: center;
}
.category-btn {
  padding: 7px 14px;
  font-size: .9rem;
  border: 1px solid #cbd5e0;
  border-radius: 8px;
  background: #f7fafc;
  color: #4a5568;
  cursor: pointer;
  transition: all .15s ease;
}
.category-btn:hover {
  border-color: #805ad5;
  color: #805ad5;
}
.category-btn.active {
  background: #805ad5;
  border-color: #805ad5;
  color: #fff;
  font-weight: 600;
}
.mode-badge {
  margin-left: 6px;
  font-size: .7rem;
  background: #ffffff40;
  border-radius: 4px;
  padding: 1px 5px;
  vertical-align: middle;
}
.genre-submenu {
  margin-top: 12px;
  padding-top: 12px;
  border-top: 1px dashed #e2e8f0;
  display: flex;
  gap: 8px;
  flex-wrap: wrap;
}
.genre-btn {
  padding: 6px 12px;
  font-size: .85rem;
  border: 1px solid #cbd5e0;
  border-radius: 999px;
  background: #fff;
  color: #4a5568;
  cursor: pointer;
  transition: all .15s ease;
}
.genre-btn:hover {
  border-color: #38a169;
  color: #38a169;
}
.genre-btn.active {
  background: #38a169;
  border-color: #38a169;
  color: #fff;
  font-weight: 600;
}
.slider-bar {
  margin-top: 12px;
  padding-top: 12px;
  border-top: 1px dashed #e2e8f0;
  display: flex;
  align-items: center;
  gap: 14px;
  flex-wrap: wrap;
}
.slider-label {
  font-size: .9rem;
  color: #4a5568;
  min-width: 220px;
}
.rank-tag {
  display: inline-block;
  margin-right: 8px;
  padding: 2px 8px;
  border-radius: 999px;
  background: #805ad5;
  color: #fff;
  font-size: .78rem;
  font-weight: 600;
}
.slider-bar input[type=range] {
  flex: 1;
  min-width: 200px;
  accent-color: #805ad5;
}
.slider-row {
  display: flex;
  align-items: center;
  gap: 10px;
  flex: 1;
  min-width: 220px;
}
.slider-row input[type=range] {
  flex: 1;
  min-width: 160px;
}
.slider-bound {
  font-size: .78rem;
  color: #a0aec0;
  white-space: nowrap;
}
.slider-reset {
  padding: 6px 14px;
  font-size: .85rem;
  border: 1px solid #cbd5e0;
  border-radius: 8px;
  background: #fff;
  color: #4a5568;
  cursor: pointer;
  transition: all .15s ease;
}
.slider-reset:hover {
  border-color: #e53e3e;
  color: #e53e3e;
}
.cinema-section {
  background: #fff;
  border: 1px solid #e2e8f0;
  border-radius: 14px;
  padding: 29px;
  box-shadow: 0 1px 3px #00000014;
}
.section-title {
  font-size: 1.8rem;
  color: #1a202c;
  margin-bottom: 24px;
  border-bottom: 2px solid #e2e8f0;
  padding-bottom: 14px;
}
.section-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 12px;
  flex-wrap: wrap;
  margin-bottom: 24px;
  border-bottom: 2px solid #e2e8f0;
  padding-bottom: 14px;
}
.section-header .section-title {
  margin-bottom: 0;
  border-bottom: none;
  padding-bottom: 0;
}
.movie-search {
  width: 260px;
  margin-left: auto;
  padding: 9px 14px;
  border: 1px solid #cbd5e0;
  border-radius: 8px;
  font-size: .95rem;
  outline: none;
  transition: border-color .15s ease;
}
.movie-search:focus {
  border-color: #3182ce;
}
.cinema-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit,minmax(360px,1fr));
  gap: 24px;
}
.cinema-block-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 12px;
  margin-bottom: 12px;
}
.booking-btn {
  padding: 7px 14px;
  font-size: .85rem;
  font-weight: 600;
  border-radius: 8px;
  background: #ed8936;
  color: #fff;
  text-decoration: none;
  white-space: nowrap;
  transition: background .15s ease;
}
.booking-btn:hover {
  background: #dd6b20;
}
.booking-btn.disabled {
  background: #cbd5e0;
  color: #fff;
  cursor: not-allowed;
  pointer-events: none;
}
.cinema-block {
  border: 1px solid #e2e8f0;
  border-radius: 10px;
  padding: 16px;
  background: #fafbfc;
}
.cinema-showtimes {
  border-top: 4px solid #e53e3e;
}
.cinema-miramar {
  border-top: 4px solid #3182ce;
}
.cinema-title {
  font-size: 1.2rem;
  color: #1a202c;
}
.status {
  color: #718096;
  font-size: .95rem;
}
.status.error {
  color: #e53e3e;
}
.movie-list {
  list-style: none;
  max-height: 480px;
  overflow-y: auto;
}
.movie-item {
  padding: 10px;
  border-bottom: 1px dashed #e2e8f0;
  cursor: pointer;
  transition: background .15s ease;
  border-radius: 6px;
}
.movie-item:hover {
  background: #edf2f7;
}
.movie-item:last-child {
  border-bottom: none;
}
.movie-title {
  font-weight: 600;
  font-size: .95rem;
  margin-bottom: 6px;
  display: flex;
  align-items: center;
  gap: 8px;
  flex-wrap: wrap;
}
.movie-sources {
  display: inline-flex;
  gap: 4px;
  margin-left: auto;
}
.source-tag {
  font-size: .82rem;
  font-weight: 600;
  color: #3182ce;
  background: #ebf8ff;
  border: 1px solid #bee3f8;
  border-radius: 999px;
  padding: 3px 10px;
  white-space: nowrap;
}
.movie-meta {
  display: flex;
  gap: 10px;
}
.movie-actions {
  display: flex;
  justify-content: flex-end;
  margin-top: 8px;
}
.watchlist-add-btn {
  padding: 6px 12px;
  font-size: .8rem;
  font-weight: 600;
  border: 1px solid #38a169;
  border-radius: 8px;
  background: #f0fff4;
  color: #38a169;
  cursor: pointer;
  transition: all .15s ease;
}
.watchlist-add-btn:hover:not(:disabled) {
  background: #38a169;
  color: #fff;
}
.watchlist-add-btn.added {
  background: #e2e8f0;
  border-color: #a0aec0;
  color: #718096;
  cursor: not-allowed;
}
.movie-poster {
  width: 66px;
  height: 98px;
  object-fit: cover;
  border-radius: 5px;
  flex-shrink: 0;
}
.movie-info {
  font-size: .85rem;
  color: #4a5568;
  line-height: 1.5;
}
.movie-field .field-label {
  color: #a0aec0;
  margin-right: 4px;
}
.movie-no-meta {
  font-size: .8rem;
  color: #a0aec0;
}
.modal-overlay {
  position: fixed;
  top: 0;
  right: 0;
  bottom: 0;
  left: 0;
  background: #1a202c99;
  display: flex;
  align-items: center;
  justify-content: center;
  padding: 24px;
  z-index: 100;
}
.modal-card {
  position: relative;
  background: #fff;
  border-radius: 14px;
  max-width: 640px;
  width: 100%;
  max-height: 88vh;
  overflow-y: auto;
  box-shadow: 0 20px 50px #00000059;
}
.modal-close {
  position: absolute;
  top: 10px;
  right: 12px;
  width: 34px;
  height: 34px;
  border: none;
  border-radius: 50%;
  background: #00000073;
  color: #fff;
  font-size: 1.3rem;
  line-height: 1;
  cursor: pointer;
  z-index: 10;
  transition: background .15s ease;
}
.modal-close:hover {
  background: #000000b3;
}
.modal-hero {
  position: relative;
  background: #1a202c;
}
.modal-backdrop {
  width: 100%;
  height: 200px;
  object-fit: cover;
  opacity: .45;
  display: block;
}
.modal-hero-content {
  position: absolute;
  top: 0;
  right: 0;
  bottom: 0;
  left: 0;
  display: flex;
  align-items: flex-end;
  gap: 16px;
  padding: 16px 20px;
  background: linear-gradient(to top,rgba(26,32,44,.9),transparent 70%);
}
.modal-poster {
  width: 96px;
  height: 144px;
  object-fit: cover;
  border-radius: 8px;
  box-shadow: 0 4px 12px #00000080;
  flex-shrink: 0;
}
.modal-poster.placeholder {
  display: flex;
  align-items: center;
  justify-content: center;
  background: #4a5568;
  color: #a0aec0;
  font-size: .85rem;
}
.modal-hero-text {
  color: #fff;
  min-width: 0;
}
.modal-title {
  font-size: 1.5rem;
  margin-bottom: 4px;
}
.modal-subtitle {
  font-size: .9rem;
  color: #cbd5e0;
  margin-bottom: 8px;
}
.modal-rating {
  font-size: 1.1rem;
}
.rating-star {
  color: #f6ad55;
}
.rating-count {
  font-size: .85rem;
  color: #cbd5e0;
  margin-left: 6px;
}
.modal-section {
  padding: 18px 22px;
  border-top: 1px solid #e2e8f0;
}
.modal-section-title {
  font-size: 1rem;
  color: #2d3748;
  margin-bottom: 10px;
}
.modal-overview {
  font-size: .95rem;
  color: #4a5568;
  line-height: 1.7;
}
.modal-fields {
  width: 100%;
  border-collapse: collapse;
  font-size: .92rem;
}
.modal-fields tr {
  border-bottom: 1px solid #edf2f7;
}
.modal-fields tr:last-child {
  border-bottom: none;
}
.modal-field-label {
  width: 45%;
  padding: 7px 10px;
  color: #718096;
  font-size: .85rem;
  vertical-align: top;
}
.modal-field-value {
  padding: 7px 10px;
  color: #2d3748;
  word-break: break-all;
}
.modal-footer {
  display: flex;
  justify-content: flex-end;
  gap: 10px;
  flex-wrap: wrap;
  padding-top: 16px;
  border-top: 1px solid #e2e8f0;
  margin-top: 16px;
}
.watchlist-block {
  width: 280px;
  flex-shrink: 0;
  display: flex;
  flex-direction: column;
  background: #fff;
  border: 1px solid #e2e8f0;
  border-radius: 12px;
  box-shadow: 0 1px 3px #00000014;
  overflow: hidden;
}
.watchlist-list {
  list-style: none;
  padding: 8px;
  display: flex;
  flex-direction: column;
  gap: 8px;
  overflow-y: auto;
  min-height: 0;
}
.watchlist-header {
  display: flex;
  align-items: center;
  gap: 8px;
  padding: 14px 16px;
  border-bottom: 1px solid #e2e8f0;
  flex-shrink: 0;
}
.watchlist-title {
  font-size: 1.05rem;
  font-weight: 600;
  color: #1a202c;
}
.watchlist-count {
  font-size: .8rem;
  font-weight: 600;
  color: #38a169;
  background: #f0fff4;
  border: 1px solid #9ae6b4;
  border-radius: 999px;
  padding: 1px 8px;
}
.watchlist-empty {
  padding: 24px 16px;
  font-size: .85rem;
  color: #a0aec0;
  line-height: 1.7;
  text-align: center;
}
.watchlist-list {
  list-style: none;
  padding: 8px;
  display: flex;
  flex-direction: column;
  gap: 8px;
}
.watchlist-item {
  border: 1px solid #e2e8f0;
  border-radius: 10px;
  padding: 10px;
  background: #fafbfc;
}
.watchlist-item-top {
  display: flex;
  gap: 10px;
  align-items: flex-start;
  margin-bottom: 8px;
}
.watchlist-poster {
  width: 48px;
  height: 71px;
  object-fit: cover;
  border-radius: 4px;
  flex-shrink: 0;
}
.watchlist-item-title {
  font-size: .9rem;
  font-weight: 600;
  color: #2d3748;
  line-height: 1.45;
}
.watchlist-remove-btn {
  display: block;
  margin-left: auto;
  padding: 5px 10px;
  font-size: .75rem;
  font-weight: 600;
  border: 1px solid #e53e3e;
  border-radius: 8px;
  background: #fff5f5;
  color: #e53e3e;
  cursor: pointer;
  transition: all .15s ease;
}
.watchlist-remove-btn:hover {
  background: #e53e3e;
  color: #fff;
}
.recent-block {
  width: 280px;
  flex-shrink: 0;
  display: flex;
  flex-direction: column;
  background: #fff;
  border: 1px solid #e2e8f0;
  border-radius: 12px;
  box-shadow: 0 1px 3px #00000014;
  overflow: hidden;
}
.recent-header {
  display: flex;
  align-items: center;
  gap: 8px;
  padding: 14px 16px;
  border-bottom: 1px solid #e2e8f0;
  flex-shrink: 0;
}
.recent-title {
  font-size: 1.05rem;
  font-weight: 600;
  color: #1a202c;
}
.recent-count {
  font-size: .8rem;
  font-weight: 600;
  color: #805ad5;
  background: #faf5ff;
  border: 1px solid #d6bcfa;
  border-radius: 999px;
  padding: 1px 8px;
}
.recent-empty {
  padding: 24px 16px;
  font-size: .85rem;
  color: #a0aec0;
  line-height: 1.7;
  text-align: center;
}
.recent-list {
  list-style: none;
  padding: 8px;
  display: flex;
  flex-direction: column;
  gap: 8px;
  overflow-y: auto;
  min-height: 0;
}
.recent-item {
  border: 1px solid #e2e8f0;
  border-radius: 10px;
  padding: 8px;
  background: #fafbfc;
}
.recent-item-main {
  display: flex;
  align-items: center;
  gap: 10px;
  width: 100%;
  background: none;
  border: none;
  padding: 0;
  cursor: pointer;
  text-align: left;
}
.recent-item-main:hover .recent-item-title {
  color: #3182ce;
}
.recent-poster {
  width: 42px;
  height: 62px;
  object-fit: cover;
  border-radius: 4px;
  flex-shrink: 0;
}
.recent-item-title {
  font-size: .88rem;
  font-weight: 600;
  color: #2d3748;
  line-height: 1.4;
  transition: color .15s ease;
}
.recent-add {
  display: block;
  margin-left: auto;
  margin-top: 8px;
}
.chat-panel {
  width: 340px;
  flex-shrink: 0;
  align-self: flex-start;
  display: flex;
  flex-direction: column;
  background: #fff;
  border: 1px solid #e2e8f0;
  border-radius: 12px;
  box-shadow: 0 1px 3px #00000014;
  height: calc(100vh - 48px);
  position: sticky;
  top: 24px;
}
.chat-header {
  display: flex;
  align-items: baseline;
  gap: 8px;
  padding: 14px 16px;
  border-bottom: 1px solid #e2e8f0;
}
.chat-title {
  font-size: 1.05rem;
  font-weight: 600;
  color: #1a202c;
}
.chat-hint {
  font-size: .78rem;
  color: #a0aec0;
}
.chat-messages {
  flex: 1;
  overflow-y: auto;
  padding: 14px 16px;
  display: flex;
  flex-direction: column;
  gap: 10px;
}
.chat-placeholder {
  font-size: .85rem;
  color: #a0aec0;
  line-height: 1.6;
}
.chat-msg {
  padding: 9px 12px;
  border-radius: 10px;
  font-size: .9rem;
  line-height: 1.55;
  max-width: 88%;
  white-space: pre-wrap;
  word-break: break-word;
}
.chat-user {
  align-self: flex-end;
  background: #3182ce;
  color: #fff;
}
.chat-assistant {
  align-self: flex-start;
  background: #edf2f7;
  color: #2d3748;
}
.chat-thinking {
  color: #a0aec0;
  font-style: italic;
}
.chat-quick-prompts {
  display: flex;
  flex-wrap: wrap;
  gap: 6px;
  padding: 10px 16px 0;
}
.quick-chip {
  font-size: .75rem;
  padding: 5px 10px;
  border: 1px solid #cbd5e0;
  border-radius: 999px;
  background: #f7fafc;
  color: #4a5568;
  cursor: pointer;
  transition: all .15s ease;
}
.quick-chip:hover:not(:disabled) {
  border-color: #3182ce;
  color: #3182ce;
  background: #ebf8ff;
}
.quick-chip:disabled {
  opacity: .5;
  cursor: not-allowed;
}
.chat-input-row {
  display: flex;
  gap: 8px;
  padding: 12px 16px;
  border-top: 1px solid #e2e8f0;
}
.chat-input {
  flex: 1;
  padding: 9px 12px;
  border: 1px solid #cbd5e0;
  border-radius: 8px;
  font-size: .9rem;
  outline: none;
}
.chat-input:focus {
  border-color: #3182ce;
}
.chat-send {
  padding: 9px 16px;
  border: none;
  border-radius: 8px;
  background: #3182ce;
  color: #fff;
  font-size: .9rem;
  font-weight: 600;
  cursor: pointer;
  transition: background .15s ease;
}
.chat-send:hover:not(:disabled) {
  background: #2b6cb0;
}
.chat-send:disabled {
  background: #a0aec0;
  cursor: not-allowed;
}
@media (max-width: 900px) {
  .page {
    flex-direction: column;
  }
  .page-left {
    width: 100%;
    position: static;
    max-height: none;
    flex-direction: row;
    flex-wrap: wrap;
  }
  .page-left .watchlist-block, .page-left .recent-block {
    flex: 1 1 280px;
    max-height: 320px;
  }
  .chat-panel {
    width: 100%;
    height: 420px;
    position: static;
  }
}
* {
  margin: 0;
  padding: 0;
  box-sizing: border-box;
}
body {
  font-family: Segoe UI,Microsoft JhengHei,sans-serif;
  background: #f4f5f7;
  color: #2d3748;
}
</style>
</head>
<body>
<div id="root">

  <div class="page">

    <!-- 左欄：待看清單 / 最近瀏覽（兩塊的內容都由 JS 填） -->
    <div class="page-left">
      <aside class="watchlist-block" id="watchlist-block"></aside>
      <aside class="recent-block" id="recent-block"></aside>
    </div>

    <!-- 中欄：標題 / 篩選列 / 電影清單 -->
    <div class="page-main">
      <h1 class="page-title">全國電影整合系統</h1>

      <div class="category-filter">
        <div class="category-filter-header">
          <span class="category-filter-title">分類篩選</span>
          <input type="text" class="movie-search" id="movie-search" placeholder="搜尋片名…">
        </div>
        <div class="category-buttons" id="category-buttons"></div>
        <!-- 下面兩個容器平常是空的（空 div 不佔高度也沒有邊框），
             按下「類型」或某個排序鍵時才會被填進去 -->
        <div id="genre-host"></div>
        <div id="slider-host"></div>
      </div>

      <section class="cinema-section">
        <div class="section-header">
          <h2 class="section-title">電影</h2>
        </div>
        <div id="movie-area">
          <ul class="movie-list" id="movie-list"></ul>
        </div>
      </section>
    </div>

    <!-- 右欄：AI 聊天 -->
    <aside class="chat-panel">
      <div class="chat-header">
        <span class="chat-title">電影推薦助手</span>
        <span class="chat-hint" id="chat-hint">（等待電影資料…）</span>
      </div>
      <div class="chat-messages" id="chat-messages"></div>
      <div class="chat-quick-prompts" id="chat-quick"></div>
      <div class="chat-input-row">
        <input type="text" class="chat-input" id="chat-input" placeholder="輸入訊息…">
        <button type="button" class="chat-send" id="chat-send">送出</button>
      </div>
    </aside>

  </div>

  <!-- 詳細資料視窗。overlay 是 position:fixed，所以放在 .page 外面
       完全不影響版面，也不會多出一個 flex 項目 -->
  <div id="modal-host"></div>

</div>
<script>
/* --------------------------------------------------------------------------
 * 前端。這支檔案不重複實作任何後端邏輯：
 * 抓片單、查 TMDB、挑最佳比對結果全都在 movieapp/ 裡，
 * 這裡只負責「跟 /api/ 要資料、畫出來、把問題轉給 /api/chat/」。
 *
 * 篩選、排序、去重放在瀏覽器做，因為那要即時反應 ——
 * 每按一次按鈕就往伺服器跑一趟太慢，而且那些資料早就在手上了。
 * -------------------------------------------------------------------------- */

const API = "/api";
const IMG_SMALL = "https://image.tmdb.org/t/p/w92";
const IMG_MEDIUM = "https://image.tmdb.org/t/p/w300";
const IMG_ORIGINAL = "https://image.tmdb.org/t/p/original";

// 兩家影城：key 同時是 /api/<key>/ 的路徑、來源代號、訂票連結的識別
const CINEMAS = [
  { key: "showtimes", label: "秀泰影城", url: "https://www.showtimes.com.tw/" },
  { key: "miramar", label: "美麗華影城", url: "https://www.miramarcinemas.tw/" },
];

// 篩選列上的五顆按鈕
const FILTER_BUTTONS = [
  { key: "adult", label: "是否成人片", type: "bool" },
  { key: "genre_ids", label: "類型", type: "genre" },
  { key: "popularity", label: "熱門度", type: "slider" },
  { key: "releaseDate", label: "上映日期", type: "slider" },
  { key: "voteAverage", label: "評分", type: "slider" },
];

const SORT_KEYS = ["popularity", "releaseDate", "voteAverage"];

const QUICK_PROMPTS = [
  "推薦一部高分電影",
  "推薦黑暗風格的電影",
  "推薦喜劇片",
  "哪部電影評分最高？",
  "推薦今天上映的電影",
  "推薦動作片",
];

// 同時發出幾個 TMDB 查詢。
//
// 一部一部排隊查，六十幾部片乘上每次約半秒的來回，要等三十秒以上；
// 全部一次射出去又會撞到 TMDB 的速率限制（429）。
// 開一個固定大小的池子，維持八個請求在路上，是這兩者之間的折衷。
const TMDB_CONCURRENCY = 8;


/* --------------------------------------------------------------------------
 * 狀態
 * -------------------------------------------------------------------------- */
const state = {
  // 每家影城各自一份：片名、補完 TMDB 的電影、載入中、錯誤
  cinemas: {},
  genres: {},
  genreError: "",
  filters: {
    adult: "off",
    genreId: null,
    popularity: { mode: "off", threshold: null },
    releaseDate: { mode: "off", threshold: null },
    voteAverage: { mode: "off", threshold: null },
  },
  genrePanelOpen: false,
  search: "",
  watchlist: loadWatchlist(),
  recent: [],
  selected: null,
  chat: { messages: [], sending: false, sessionId: newSessionId() },
};

for (const cinema of CINEMAS) {
  state.cinemas[cinema.key] = {
    titles: [], movies: [], loading: true, error: "", enriching: false, tmdbError: "",
  };
}

function loadWatchlist() {
  try {
    return JSON.parse(localStorage.getItem("watchlist") || "[]") || [];
  } catch {
    return [];
  }
}

function saveWatchlist() {
  try {
    localStorage.setItem("watchlist", JSON.stringify(state.watchlist));
  } catch {
    /* 無痕視窗或關掉儲存空間時會丟例外，不影響其他功能 */
  }
}

function newSessionId() {
  return (typeof crypto !== "undefined" && crypto.randomUUID)
    ? crypto.randomUUID()
    : String(Math.random()).slice(2);
}


/* --------------------------------------------------------------------------
 * 小工具
 * -------------------------------------------------------------------------- */
const $ = (id) => document.getElementById(id);

/** 建立元素。attrs 裡的 class/text 是捷徑，on* 會掛成事件。 */
function el(tag, attrs, ...children) {
  const node = document.createElement(tag);
  let text = null;
  for (const [key, value] of Object.entries(attrs || {})) {
    if (value == null || value === false) continue;
    if (key === "class") node.className = value;
    else if (key === "text") text = value;
    else if (key.startsWith("on")) node.addEventListener(key.slice(2).toLowerCase(), value);
    else node.setAttribute(key, value === true ? "" : String(value));
  }
  setChildren(node, text, ...children);
  return node;
}

/** 把 children 換掉。跟 el() 一樣會攤平陣列、略過 null/false，
 *  所以 `條件 && el(...)` 這種寫法在這裡也安全。
 *  （原生的 replaceChildren 不會過濾，會把 false 直接印成 "false"。） */
function setChildren(node, ...children) {
  node.replaceChildren();
  for (const child of children.flat(Infinity)) {
    if (child == null || child === false || child === "") continue;
    node.appendChild(child instanceof Node ? child : document.createTextNode(String(child)));
  }
}

/** 一部電影的唯一身分：有 TMDB id 就用 id，沒有就退回片名。
 *  和後端 merge.movie_key() 是同一個想法。 */
function movieId(movie) {
  const id = movie.meta && movie.meta.id;
  return id != null ? id : movie.title;
}

function sameMovie(a, b) {
  return movieId(a) === movieId(b);
}

function inWatchlist(movie) {
  return state.watchlist.some((item) => sameMovie(item, movie));
}

/** "2026-08-07" -> 可以直接比大小的月份序號。和後端 merge._release_month() 一致。 */
function releaseMonth(value) {
  if (!value) return null;
  const match = /^(\d{4})-(\d{1,2})/.exec(String(value).trim());
  return match ? Number(match[1]) * 12 + Number(match[2]) : null;
}

/** 月份序號 -> "2026年8月" */
function monthLabel(value) {
  if (value == null) return "";
  return `${Math.floor((value - 1) / 12)}年${((value - 1) % 12) + 1}月`;
}

function modeLabel(mode) {
  return mode === "desc" ? "高→低" : "低→高";
}

function genreNames(ids) {
  return (ids || []).map((id) => state.genres[id] || id).join("、");
}


/* --------------------------------------------------------------------------
 * 抓資料
 *
 * 流程和第 01~02 章一模一樣，只是換到瀏覽器這一側跑：
 *     /api/<影城>/          -> 一串片名
 *     /api/tmdb/search/     -> 每個片名補上 TMDB 資料
 *
 * 為什麼不做成一支「一次回傳全部」的 API？因為那樣畫面會整整空白十幾秒。
 * 分開之後，片單一到就先畫出來，TMDB 的海報和評分再一批一批補上去。
 * -------------------------------------------------------------------------- */
async function loadCinema(cinema) {
  const store = state.cinemas[cinema.key];

  // 第一步：片名
  try {
    const response = await fetch(`${API}/${cinema.key}/`);
    const data = await response.json();
    if (response.ok) store.titles = data.movies || [];
    else store.error = data.error || "載入失敗";
  } catch {
    store.error = "無法連線到後端伺服器";
  }
  store.loading = false;
  render();

  if (store.error || store.titles.length === 0) return;

  // 第二步：逐部補 TMDB 資料
  await enrichTitles(cinema, store);
}

async function enrichTitles(cinema, store) {
  const titles = store.titles;

  // 結果先寫進固定位置再過濾，這樣不管誰先回來，
  // 畫面上的順序永遠跟影城給的片單順序一致。
  const slots = new Array(titles.length).fill(null);
  let cursor = 0;

  store.enriching = true;
  render();

  async function worker() {
    while (cursor < titles.length) {
      const index = cursor++;
      const title = titles[index];
      try {
        const url = `${API}/tmdb/search/?query=${encodeURIComponent(title)}&language=zh-TW`;
        const response = await fetch(url);
        const data = await response.json();
        if (response.ok) {
          // 只取第一筆 —— 後端已經把最像的那部排到最前面了（第 02 章的陷阱）
          const meta = (data.results || [])[0] || null;
          if (meta) slots[index] = { title, meta, source: cinema.key };
        } else if (!store.tmdbError) {
          store.tmdbError = data.error || "TMDB 查詢失敗";
        }
      } catch {
        if (!store.tmdbError) store.tmdbError = "無法連線到 TMDB";
      }
      store.movies = slots.filter(Boolean);
      scheduleRender();
    }
  }

  const size = Math.min(TMDB_CONCURRENCY, titles.length);
  await Promise.all(Array.from({ length: size }, worker));

  store.enriching = false;
  render();
}

async function loadGenres() {
  try {
    const response = await fetch(`${API}/tmdb/genres/?language=zh-TW`);
    const data = await response.json();
    if (!response.ok) {
      state.genreError = data.error || "類型載入失敗";
    } else {
      const names = {};
      for (const genre of data.genres || []) names[genre.id] = genre.name;
      state.genres = names;
    }
  } catch {
    state.genreError = "無法載入類型";
  }
  render();
}


/* --------------------------------------------------------------------------
 * 整合、篩選、排序
 *
 * 這一段是第 03 章 merge.py 的瀏覽器版本，規則完全一樣：
 * 用 TMDB id 當共同身分證跨影城去重，查不到 id 的才退回用片名。
 * -------------------------------------------------------------------------- */
function mergedMovies() {
  const merged = new Map();
  for (const cinema of CINEMAS) {
    for (const movie of state.cinemas[cinema.key].movies) {
      const id = movie.meta && movie.meta.id;
      const key = id != null ? `tmdb:${id}` : `title:${movie.title}`;
      const existing = merged.get(key);
      if (existing) {
        if (!existing.sources.includes(movie.source)) existing.sources.push(movie.source);
      } else {
        merged.set(key, { ...movie, sources: [movie.source] });
      }
    }
  }
  return Array.from(merged.values());
}

/** 滑桿的刻度要跟著實際資料走，不能寫死。 */
function bounds(movies) {
  const populars = movies.map((m) => (m.meta && m.meta.popularity) || 0);
  const years = movies
    .map((m) => Number.parseInt(m.meta && m.meta.release_date, 10))
    .filter(Boolean);
  return {
    popularityMax: Math.max(10, ...populars),
    yearMin: years.length ? Math.min(...years) : 1990,
    yearMax: Math.max(new Date().getFullYear(), ...years),
  };
}

function applyFilters(movies, filters) {
  const kept = movies.filter((movie) => {
    const meta = movie.meta;
    if (filters.adult === "true" && !(meta && meta.adult)) return false;
    if (filters.adult === "false" && meta && meta.adult) return false;
    if (filters.genreId && !((meta && meta.genre_ids) || []).includes(filters.genreId)) return false;
    if (filters.popularity.threshold != null
        && ((meta && meta.popularity) ?? 0) < filters.popularity.threshold) return false;
    if (filters.voteAverage.threshold != null
        && ((meta && meta.vote_average) ?? 0) < filters.voteAverage.threshold) return false;
    if (filters.releaseDate.threshold != null) {
      const month = releaseMonth(meta && meta.release_date);
      if (month == null || month < filters.releaseDate.threshold) return false;
    }
    return true;
  });

  // 三個排序鍵一次只會有一個生效
  const active = SORT_KEYS.find((key) => filters[key].mode !== "off");
  if (!active) return kept;

  const field = active === "releaseDate" ? "release_date"
    : active === "voteAverage" ? "vote_average" : "popularity";
  const direction = filters[active].mode === "asc" ? 1 : -1;

  return [...kept].sort((a, b) => {
    const left = a.meta && a.meta[field];
    const right = b.meta && b.meta[field];
    // 沒有值的一律沉到最後，不管是升冪還是降冪
    if (left == null || left === "") return 1;
    if (right == null || right === "") return -1;
    return typeof left === "number"
      ? (left - right) * direction
      : (Date.parse(left) - Date.parse(right)) * direction;
  });
}

function searchMovies(movies, keyword) {
  const query = keyword.trim().toLowerCase();
  if (!query) return movies;
  return movies.filter((movie) =>
    movie.title.toLowerCase().includes(query)
    || ((movie.meta && movie.meta.title) || "").toLowerCase().includes(query));
}

/** 目前畫面上實際看得到的電影 —— 聊天的 prompt 也是拿這一份。 */
function visibleMovies() {
  return searchMovies(applyFilters(mergedMovies(), state.filters), state.search);
}

/** 把畫面上的電影塞進系統提示。
 *  這是第 04 章 gemini.build_system_prompt() 的同一個想法：
 *  模型本身不知道今天上映什麼，是我們把資料餵給它的（最陽春的 RAG）。 */
function buildSystemPrompt(movies) {
  if (movies.length === 0) return "";
  const lines = movies.map((movie, index) => {
    const meta = movie.meta;
    const parts = [
      meta && meta.title ? `片名: ${meta.title}` : "",
      meta && meta.release_date ? `上映日期: ${meta.release_date}` : "",
      meta && meta.vote_average != null ? `評分: ${meta.vote_average.toFixed(1)}` : "",
      meta && meta.genre_ids && meta.genre_ids.length ? `類型: ${genreNames(meta.genre_ids)}` : "",
      meta && meta.overview ? `簡介: ${meta.overview.slice(0, 200)}` : "",
    ].filter(Boolean);
    return `${index + 1}. ${movie.title}${parts.length ? `（${parts.join("，")}）` : ""}`;
  });
  return "你是電影推薦助手，請用繁體中文回答。以下是目前頁面顯示的電影資料，"
    + "請依據這些資料回答使用者的問題（例如推薦、比較、說明）：\n"
    + lines.join("\n");
}


/* --------------------------------------------------------------------------
 * 操作
 * -------------------------------------------------------------------------- */
function cycleAdult() {
  const order = { off: "true", true: "false", false: "off" };
  state.filters.adult = order[state.filters.adult];
  render();
}

function toggleGenrePanel() {
  state.genrePanelOpen = !state.genrePanelOpen;
  if (!state.genrePanelOpen) state.filters.genreId = null;  // 收起來就順手清掉選擇
  render();
}

function selectGenre(id) {
  state.filters.genreId = state.filters.genreId === id ? null : id;
  render();
}

/** 排序鍵按一下換一種：不排 -> 高到低 -> 低到高 -> 不排。
 *  同時把另外兩個鍵關掉，因為一次只排一個欄位。 */
function cycleSlider(key) {
  const current = state.filters[key].mode;
  const next = current === "off" ? "desc" : current === "desc" ? "asc" : "off";
  for (const other of SORT_KEYS) {
    state.filters[other] = other === key
      ? { mode: next, threshold: next === "off" ? null : state.filters[key].threshold }
      : { mode: "off", threshold: null };
  }
  render();
}

function setThreshold(key, value) {
  state.filters[key].threshold = value;
  render();
}

function resetThreshold(key) {
  state.filters[key].threshold = null;
  render();
}

function addToWatchlist(movie) {
  if (inWatchlist(movie)) return;
  state.watchlist = [...state.watchlist, movie];
  saveWatchlist();
  render();
}

function removeFromWatchlist(movie) {
  state.watchlist = state.watchlist.filter((item) => !sameMovie(item, movie));
  saveWatchlist();
  render();
}

/** 點一部電影：開詳細資料，同時記進「最近瀏覽」（只留最新 10 筆）。 */
function selectMovie(movie) {
  state.selected = movie;
  state.recent = [movie, ...state.recent.filter((item) => !sameMovie(item, movie))].slice(0, 10);
  render();
}

function closeModal() {
  state.selected = null;
  render();
}

async function sendChat(text) {
  const message = (text || "").trim();
  if (!message || state.chat.sending) return;

  state.chat.messages = [...state.chat.messages, { role: "user", content: message }];
  state.chat.sending = true;
  $("chat-input").value = "";
  render();

  try {
    const response = await fetch(`${API}/chat/`, {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({
        system: buildSystemPrompt(visibleMovies()),
        messages: state.chat.messages,
        session_id: state.chat.sessionId,
      }),
    });
    const data = await response.json();
    state.chat.messages = [...state.chat.messages, {
      role: "assistant",
      content: data.reply || data.error || "（無回應）",
    }];
  } catch {
    state.chat.messages = [...state.chat.messages, {
      role: "assistant",
      content: "無法連線聊天服務",
    }];
  }
  state.chat.sending = false;
  render();
}


/* --------------------------------------------------------------------------
 * 畫面
 *
 * 沒有框架，所以自己處理「不要重畫沒變的東西」：
 * 每一區先算一個簽章，跟上次一樣就跳過。
 * 這不只是省效能 —— 一直重建 <img> 會讓海報閃爍，
 * 而且重建 <input> 會把游標和拖曳中的滑桿弄掉。
 * -------------------------------------------------------------------------- */
const signatures = {};

function changed(name, value) {
  if (signatures[name] === value) return false;
  signatures[name] = value;
  return true;
}

let renderQueued = false;

/** 補資料時每回來一筆就重畫一次太浪費，合併到下一個畫面更新再處理。 */
function scheduleRender() {
  if (renderQueued) return;
  renderQueued = true;
  requestAnimationFrame(() => {
    renderQueued = false;
    render();
  });
}

function render() {
  const merged = mergedMovies();
  const visible = searchMovies(applyFilters(merged, state.filters), state.search);
  const limits = bounds(merged);

  renderWatchlist();
  renderRecent();
  renderCategoryButtons();
  renderGenreSubmenu();
  renderSliderBar(limits);
  renderMovieArea(visible);
  renderChat(visible);
  renderModal();
}


/* ---------- 左欄：待看清單 ---------- */
function renderWatchlist() {
  const list = state.watchlist;
  if (!changed("watchlist", list.map(movieId).join("|"))) return;

  const host = $("watchlist-block");
  setChildren(host,
    el("div", { class: "watchlist-header" },
      el("span", { class: "watchlist-title", text: "待看電影清單" }),
      el("span", { class: "watchlist-count", text: list.length })),
    list.length === 0
      ? el("div", { class: "watchlist-empty" }, "尚無待看電影", el("br"), "點擊電影的「加入待看清單」按鈕")
      : el("ul", { class: "watchlist-list" }, list.map((movie) => {
          const meta = movie.meta;
          return el("li", { class: "watchlist-item" },
            el("div", { class: "watchlist-item-top" },
              meta && meta.poster_path && el("img", {
                src: IMG_SMALL + meta.poster_path, alt: "",
                class: "watchlist-poster", loading: "lazy",
              }),
              el("div", { class: "watchlist-item-title", text: movie.title })),
            el("button", {
              type: "button", class: "watchlist-remove-btn", text: "刪除待看電影",
              onClick: () => removeFromWatchlist(movie),
            }));
        })));
}


/* ---------- 左欄：最近瀏覽 ---------- */
function renderRecent() {
  const list = state.recent;
  const signature = list.map(movieId).join("|") + "#" + state.watchlist.map(movieId).join("|");
  if (!changed("recent", signature)) return;

  const host = $("recent-block");
  setChildren(host,
    el("div", { class: "recent-header" },
      el("span", { class: "recent-title", text: "最近瀏覽電影" }),
      el("span", { class: "recent-count" }, list.length, "/10")),
    list.length === 0
      ? el("div", { class: "recent-empty", text: "點擊電影後會記錄在此" })
      : el("ul", { class: "recent-list" }, list.map((movie) => {
          const meta = movie.meta;
          const added = inWatchlist(movie);
          return el("li", { class: "recent-item" },
            el("button", {
              type: "button", class: "recent-item-main",
              onClick: () => selectMovie(movie),
            },
              meta && meta.poster_path && el("img", {
                src: IMG_SMALL + meta.poster_path, alt: "",
                class: "recent-poster", loading: "lazy",
              }),
              el("span", { class: "recent-item-title", text: movie.title })),
            el("button", {
              type: "button",
              class: `watchlist-add-btn recent-add${added ? " added" : ""}`,
              disabled: added,
              text: added ? "已加入" : "加入待看",
              onClick: () => addToWatchlist(movie),
            }));
        })));
}


/* ---------- 篩選列：五顆按鈕 ---------- */
function renderCategoryButtons() {
  const filters = state.filters;
  const signature = JSON.stringify(filters) + "#" + state.genrePanelOpen;
  if (!changed("categoryButtons", signature)) return;

  const adultBadge = filters.adult === "off" ? "" : filters.adult;

  setChildren($("category-buttons"), ...FILTER_BUTTONS.map((button) => {
    let active;
    let badge = "";

    if (button.type === "bool") {
      active = filters.adult !== "off";
      badge = adultBadge;
    } else if (button.type === "genre") {
      active = state.genrePanelOpen;
    } else {
      const filter = filters[button.key];
      active = filter.mode !== "off" || filter.threshold != null;
      const parts = [];
      if (filter.mode !== "off") parts.push(modeLabel(filter.mode));
      if (filter.threshold != null) parts.push(`≥${filter.threshold}`);
      badge = parts.join(" ");
    }

    return el("button", {
      type: "button",
      class: `category-btn${active ? " active" : ""}`,
      onClick: () => {
        if (button.type === "bool") cycleAdult();
        else if (button.type === "genre") toggleGenrePanel();
        else cycleSlider(button.key);
      },
    }, button.label, badge && el("span", { class: "mode-badge", text: badge }));
  }));
}


/* ---------- 篩選列：類型子選單 ---------- */
function renderGenreSubmenu() {
  const open = state.genrePanelOpen;
  const signature = `${open}|${state.filters.genreId}|${state.genreError}|${Object.keys(state.genres).length}`;
  if (!changed("genreSubmenu", signature)) return;

  const host = $("genre-host");
  if (!open) {
    setChildren(host);
    return;
  }

  setChildren(host,
    el("div", { class: "genre-submenu" },
    state.genreError && el("span", { class: "status error", text: state.genreError }),
    Object.entries(state.genres).map(([id, name]) => el("button", {
      type: "button",
      class: `genre-btn${Number(state.filters.genreId) === Number(id) ? " active" : ""}`,
      text: name,
      onClick: () => selectGenre(Number(id)),
    }))));
}


/* ---------- 篩選列：門檻滑桿 ---------- */
//
// 滑桿只有在「有排序鍵開著」的時候出現。
// 拖曳中每移動一格就會觸發重畫，這時候絕對不能重建 <input>，
// 不然滑鼠會跟拖曳中的滑桿斷開 —— 所以同一個鍵時只更新數值。
function renderSliderBar(limits) {
  const key = SORT_KEYS.find((name) => state.filters[name].mode !== "off");
  const host = $("slider-host");

  if (!key) {
    if (changed("sliderKey", "")) setChildren(host);
    return;
  }

  const spec = sliderSpec(key, limits);
  const rebuild = changed("sliderKey", key);

  if (rebuild) {
    setChildren(host,
    el("div", { class: "slider-bar" },
      el("label", { class: "slider-label", id: "slider-label" }),
      el("div", { class: "slider-row" },
        el("span", { class: "slider-bound", id: "slider-min" }),
        el("input", {
          type: "range", id: "slider-input",
          onInput: (event) => setThreshold(key, Number(event.target.value)),
        }),
        el("span", { class: "slider-bound", id: "slider-max" })),
      el("button", {
        type: "button", class: "slider-reset", text: "清除門檻",
        onClick: () => resetThreshold(key),
      })));
  }

  const filter = state.filters[key];
  setChildren($("slider-label"),
    filter.mode !== "off" && el("span", { class: "rank-tag" }, "ranking ", modeLabel(filter.mode)),
    spec.label);
  $("slider-min").textContent = spec.minLabel;
  $("slider-max").textContent = spec.maxLabel;

  const input = $("slider-input");
  input.min = spec.min;
  input.max = spec.max;
  input.step = spec.step;
  if (Number(input.value) !== spec.value) input.value = spec.value;
}

function sliderSpec(key, limits) {
  const threshold = state.filters[key].threshold;

  if (key === "popularity") {
    const max = Math.ceil(limits.popularityMax);
    const value = threshold ?? 0;
    return {
      label: ["熱門度 ≥ ", value, "（低於此值忽略）"],
      min: 0, max, step: 1, value, minLabel: "0", maxLabel: String(max),
    };
  }

  if (key === "voteAverage") {
    const value = threshold ?? 0;
    return {
      label: ["評分 ≥ ", value, "（低於此值忽略）"],
      min: 0, max: 10, step: 0.1, value, minLabel: "0", maxLabel: "10",
    };
  }

  // releaseDate：滑桿上跑的是月份序號，顯示的時候才換回「幾年幾月」
  const min = limits.yearMin * 12 + 1;
  const max = limits.yearMax * 12 + 12;
  const value = threshold ?? min;
  return {
    label: ["上映日期 ≥ ", monthLabel(value), "（早於此月忽略）"],
    min, max, step: 1, value,
    minLabel: monthLabel(min), maxLabel: monthLabel(max),
  };
}


/* ---------- 中欄：狀態訊息 + 電影清單 ---------- */
//
// 清單用「同一部電影就重用同一個 <li>」的方式更新。
// 補資料的過程中這個函式會被呼叫幾十次，每次都整份重建的話
// 海報會不停重新載入而閃爍，捲軸位置也會被拉回頂端。
const movieNodes = new Map();

function renderMovieArea(visible) {
  const stores = CINEMAS.map((cinema) => state.cinemas[cinema.key]);
  const loading = stores.some((store) => store.loading);
  const errors = stores.map((store) => store.error).filter(Boolean);
  const tmdbErrors = stores.map((store) => store.tmdbError).filter(Boolean);
  const enriching = stores.some((store) => store.enriching);

  const notices = [];
  if (loading) {
    notices.push(["status", "載入中…"]);
  } else if (errors.length) {
    notices.push(["status error", errors.join("；")]);
  } else {
    // 影城本身沒掛，但 TMDB 可能單獨出事；出事了照樣把拿得到的電影顯示出來
    if (tmdbErrors.length) notices.push(["status error", tmdbErrors.join("；")]);
    if (enriching) notices.push(["status", "查詢 TMDB 資料中…"]);
    if (!enriching && visible.length === 0) notices.push(["status", "目前沒有電影資料"]);
  }

  if (changed("notices", JSON.stringify(notices))) {
    const area = $("movie-area");
    for (const stale of area.querySelectorAll("p.status")) stale.remove();
    const list = $("movie-list");
    for (const [className, text] of notices) {
      area.insertBefore(el("p", { class: className, text }), list);
    }
  }

  reconcileMovieList(visible);
}

function reconcileMovieList(visible) {
  const list = $("movie-list");
  const wanted = visible.map(movieNode);

  // 標準的 keyed 比對：把需要的節點依序搬到定位，剩下的刪掉。
  // insertBefore 對已經在畫面上的節點是「搬移」而不是「重建」。
  let cursor = list.firstChild;
  for (const node of wanted) {
    if (cursor === node) {
      cursor = cursor.nextSibling;
      continue;
    }
    list.insertBefore(node, cursor);
  }
  while (cursor) {
    const next = cursor.nextSibling;
    list.removeChild(cursor);
    cursor = next;
  }
}

function movieNode(movie) {
  const id = movieId(movie);
  const signature = [
    movie.title,
    (movie.sources || []).join(","),
    inWatchlist(movie),
    Object.keys(state.genres).length,
  ].join("|");

  const cached = movieNodes.get(id);
  if (cached && cached.signature === signature) return cached.node;

  const node = buildMovieItem(movie);
  movieNodes.set(id, { signature, node });
  return node;
}

function buildMovieItem(movie) {
  const meta = movie.meta;
  const labels = (movie.sources || []).map((key) => {
    const cinema = CINEMAS.find((item) => item.key === key);
    return cinema ? cinema.label : key;
  });
  const added = inWatchlist(movie);

  const info = [];
  if (meta && meta.title && meta.title !== movie.title) {
    info.push(field("片名:", meta.title));
  }
  if (meta) {
    info.push(field("上映日期:", meta.release_date || "-"));
    info.push(field("評分:", meta.vote_average ? `★ ${meta.vote_average.toFixed(1)}` : "-"));
    if (meta.genre_ids && meta.genre_ids.length) info.push(field("類型:", genreNames(meta.genre_ids)));
    if (meta.original_language) info.push(field("語言:", meta.original_language));
  }

  return el("li", { class: "movie-item", onClick: () => selectMovie(movie) },
    el("div", { class: "movie-title" },
      movie.title,
      labels.length > 0 && el("span", { class: "movie-sources" },
        labels.map((label) => el("span", { class: "source-tag", text: label })))),
    meta && el("div", { class: "movie-meta" },
      meta.poster_path && el("img", {
        src: IMG_SMALL + meta.poster_path, alt: "", class: "movie-poster", loading: "lazy",
      }),
      el("div", { class: "movie-info" }, info)),
    el("div", { class: "movie-actions" },
      el("button", {
        type: "button",
        class: `watchlist-add-btn${added ? " added" : ""}`,
        disabled: added,
        text: added ? "已加入待看清單" : "加入待看清單",
        onClick: (event) => {
          event.stopPropagation();   // 不要順便把詳細資料視窗打開
          addToWatchlist(movie);
        },
      })));
}

function field(label, value) {
  return el("div", { class: "movie-field" },
    el("span", { class: "field-label", text: label }), " ", value);
}


/* ---------- 右欄：聊天 ---------- */
function renderChat(visible) {
  const disabled = visible.length === 0;
  const { messages, sending } = state.chat;

  $("chat-hint").style.display = disabled ? "" : "none";

  if (changed("chatMessages", `${messages.length}|${sending}`)) {
    setChildren($("chat-messages"),
      messages.length === 0 && el("div", { class: "chat-placeholder" },
        "輸入問題，我會根據目前顯示的電影資料回答，例如「推薦評分最高的電影」"),
      messages.map((message) => el("div", {
        class: `chat-msg chat-${message.role}`, text: message.content,
      })),
      sending && el("div", { class: "chat-msg chat-assistant chat-thinking", text: "思考中…" }));
    $("chat-messages").scrollTop = $("chat-messages").scrollHeight;
  }

  if (changed("chatQuick", `${disabled}|${sending}`)) {
    setChildren($("chat-quick"), ...QUICK_PROMPTS.map((prompt) => el("button", {
      type: "button", class: "quick-chip", text: prompt,
      disabled: sending || disabled,
      onClick: () => sendChat(prompt),
    })));
  }

  $("chat-input").disabled = sending;
  updateSendButton();
}

function updateSendButton() {
  $("chat-send").disabled = state.chat.sending || !$("chat-input").value.trim();
}


/* ---------- 詳細資料視窗 ---------- */
function renderModal() {
  const movie = state.selected;
  if (!changed("modal", movie ? String(movieId(movie)) : "")) return;

  const host = $("modal-host");
  setChildren(host);
  if (!movie) return;

  const meta = movie.meta;
  const rows = [
    ["片名 (title)", meta && meta.title],
    ["原始片名 (original_title)", meta && meta.original_title],
    ["是否成人片 (adult)", tick(meta && meta.adult)],
    ["是否為影片 (video)", tick(meta && meta.video)],
    ["類型 (genre_ids)", meta && meta.genre_ids && meta.genre_ids.length ? genreNames(meta.genre_ids) : "-"],
    ["原始語言 (original_language)", (meta && meta.original_language) || "-"],
    ["上映日期 (release_date)", (meta && meta.release_date) || "-"],
    ["評分 (vote_average)", meta && meta.vote_average != null ? `★ ${meta.vote_average.toFixed(1)}` : "-"],
    ["評分人數 (vote_count)", (meta && meta.vote_count) ?? "-"],
    ["熱門度 (popularity)", meta && meta.popularity != null ? meta.popularity.toFixed(2) : "-"],
  ];

  host.appendChild(el("div", { class: "modal-overlay", onClick: closeModal },
    el("div", { class: "modal-card", onClick: (event) => event.stopPropagation() },
      el("button", {
        type: "button", class: "modal-close", "aria-label": "關閉", text: "×", onClick: closeModal,
      }),

      el("div", { class: "modal-hero" },
        meta && meta.backdrop_path && el("img", {
          src: IMG_ORIGINAL + meta.backdrop_path, alt: "", class: "modal-backdrop",
        }),
        el("div", { class: "modal-hero-content" },
          meta && meta.poster_path
            ? el("img", { src: IMG_MEDIUM + meta.poster_path, alt: "", class: "modal-poster" })
            : el("div", { class: "modal-poster placeholder", text: "無海報" }),
          el("div", { class: "modal-hero-text" },
            el("h2", { class: "modal-title", text: movie.title }),
            meta && meta.title && meta.title !== movie.title
              && el("div", { class: "modal-subtitle" }, "TMDB 片名：", meta.title),
            el("div", { class: "modal-rating" },
              el("span", { class: "rating-star", text: "★" }), " ",
              meta && meta.vote_average != null ? meta.vote_average.toFixed(1) : "-",
              el("span", { class: "rating-count" },
                "(", String((meta && meta.vote_count) ?? 0), " 人評分)"))))),

      meta && meta.overview && el("div", { class: "modal-section" },
        el("h3", { class: "modal-section-title", text: "劇情簡介 (overview)" }),
        el("p", { class: "modal-overview", text: meta.overview })),

      el("div", { class: "modal-section" },
        el("h3", { class: "modal-section-title", text: "TMDB 資料" }),
        el("table", { class: "modal-fields" },
          el("tbody", {}, rows.map(([label, value]) => el("tr", {},
            el("td", { class: "modal-field-label", text: label }),
            el("td", { class: "modal-field-value", text: value == null ? "" : String(value) })))))),

      // 只有真的在那家影城上映才點得下去
      el("div", { class: "modal-footer" }, CINEMAS.map((cinema) => {
        const showing = (movie.sources || []).includes(cinema.key);
        return el("a", {
          href: showing ? cinema.url : null,
          target: "_blank", rel: "noopener noreferrer",
          class: `booking-btn${showing ? "" : " disabled"}`,
          "aria-disabled": String(!showing),
          onClick: (event) => { if (!showing) event.preventDefault(); },
        }, "前往", cinema.label, "訂票");
      })))));
}

function tick(value) {
  return value === true ? "是" : value === false ? "否" : "-";
}


/* --------------------------------------------------------------------------
 * 啟動
 * -------------------------------------------------------------------------- */
$("movie-search").addEventListener("input", (event) => {
  state.search = event.target.value;
  render();
});

$("chat-send").addEventListener("click", () => sendChat($("chat-input").value));

$("chat-input").addEventListener("input", updateSendButton);
$("chat-input").addEventListener("keydown", (event) => {
  if (event.key === "Enter") sendChat($("chat-input").value);
});

document.addEventListener("keydown", (event) => {
  if (event.key === "Escape" && state.selected) closeModal();
});

render();

// 類型對照表和兩家影城同時開始抓，彼此不用等
loadGenres();
for (const cinema of CINEMAS) loadCinema(cinema);
</script>
</body>
</html>


---
## 6 ・ 啟動服務

`runserver` 會一直佔著不放，直接在 cell 裡跑會把 kernel 卡死。

解法是用 `subprocess` 開一個**獨立的行程**。這比在 kernel 裡開執行緒好：

- kernel 重開不影響它
- 重跑這一格就是重啟服務，語意單純
- 不會有 Django 設定被重複初始化的問題

加 `--noreload` 是因為 Django 的自動重載會多開一個子行程，
在 Windows 上關不乾淨，會留下佔著 port 的孤兒行程。

> **做完這一章記得跑最後一格把服務關掉。**
> 如果你關掉 kernel 卻沒關服務，那個行程會變成孤兒繼續佔著 8008，
> 到時候只能用工作管理員或 `taskkill` 處理。

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

PORT = 8008
SERVER_DIR = (Path.cwd() / ".." / "server").resolve()

# 重跑這一格會先把舊的關掉
if "server" in globals():
    server.terminate()
    server.wait(timeout=10)
    print("已關閉先前的服務")

env = dict(os.environ)
env["PYTHONIOENCODING"] = "utf-8"

# 想看 Django 的完整輸出（除錯用），把 stdout 改成 open("../server.log", "w")
server = subprocess.Popen(
    [sys.executable, "manage.py", "runserver", str(PORT), "--noreload"],
    cwd=str(SERVER_DIR), env=env,
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
time.sleep(3)

print(f"服務啟動中（PID {server.pid}）→ http://127.0.0.1:{PORT}")
print("還在執行嗎？", "是" if server.poll() is None else f"否，結束碼 {server.returncode}")

---
## 7 ・ 測試每一支 API

服務起來了，逐一確認端點都正常。

`/api/movies/` 會跑十幾秒 —— 它要抓兩家影城、再對每一部片查一次 TMDB。
第 2 章的 `workers=8` 平行查詢就是在這裡發揮作用的。

In [ ]:
import json
import urllib.parse
import urllib.request
from movieapp.config import pad

BASE = f"http://127.0.0.1:{PORT}"


def call(path, payload=None):
    url = BASE + path
    if payload is None:
        request = urllib.request.Request(url)
    else:
        request = urllib.request.Request(
            url, data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            return response.status, json.loads(response.read().decode("utf-8"))
    except Exception as exc:
        body = exc.read().decode("utf-8")[:120] if hasattr(exc, "read") else str(exc)
        return 0, {"error": body}


checks = [
    "/api/time/",
    "/api/movies/",
    "/api/showtimes/",
    "/api/miramar/",
    "/api/tmdb/genres/",
    "/api/tmdb/search/?query=" + urllib.parse.quote("藍色監獄"),
]

for path in checks:
    status, data = call(path)
    key = list(data)[0]
    value = data[key]
    size = len(value) if isinstance(value, (list, dict)) else value
    print(f"  {pad(urllib.parse.unquote(path)[:44], 46)} {status}  {key}={size}")

`/api/movies/` 回來的 `movies=` 那個數字，會比
`showtimes` 加 `miramar` 的總和**少** —— 因為兩家都有的片被合併成一筆了。
那就是第 3 章 `merge_sources()` 在服務裡實際生效的證據。

In [ ]:
status, data = call("/api/movies/")
both = [m for m in data["movies"] if len(m["sources"]) > 1]

print(f"整合後 {len(data['movies'])} 部，其中 {len(both)} 部兩家都有：")
for movie in both[:5]:
    print(f"  {movie['meta']['title']}")
print()
print("errors:", data["errors"] or "無")

注意 `/api/tmdb/search/?query=藍色監獄` 回來的第一筆 ——
第 2 章那個陷阱在這裡被服務層擋掉了：

In [ ]:
status, data = call("/api/tmdb/search/?query=" + urllib.parse.quote("藍色監獄"))
for movie in data["results"][:3]:
    print(f"  {movie['title']!r}  ({movie.get('release_date') or '未定'})")

In [ ]:
# 聊天端點（POST）
status, data = call("/api/chat/", {
    "messages": [{"role": "user", "content": "推薦一部高分電影"}],
    "system": "你是電影推薦助手，請用繁體中文回答。",
    "session_id": "notebook-05",
})
print(f"HTTP {status}")
print((data.get("reply") or data.get("error"))[:300])

---
## 8 ・ 看成品

用 `IFrame` 把剛剛寫出來的網頁直接嵌進 notebook。

這個網頁做的事情就是把你前五章寫的東西串起來：
抓片單 → 補 TMDB 資料 → 合併去重 → 篩選 → 交給 AI 聊天。

In [ ]:
from IPython.display import IFrame

IFrame(f"http://127.0.0.1:{PORT}", width="100%", height=820)

> 如果 IFrame 是空白的，直接用瀏覽器開 <http://127.0.0.1:8008>。
> 有些 Jupyter 設定會擋掉內嵌的 localhost 網頁。
>
> 版面大約 0.1 秒就會出現，電影會在接下來幾秒內一批一批補上來。

畫面上可以操作的東西：

- 左側：待看清單、最近瀏覽（存在瀏覽器的 `localStorage` 裡）
- 中間：搜尋、篩選（成人片、類型）、排序與門檻（熱門度、上映日期、評分）、電影清單
- 右側：AI 聊天助手，六個快捷問題
- 點電影卡片會開詳細資料，裡面有 TMDB 的每一個欄位和訂票連結

排序按鈕按一下換一種：不排 → 高到低 → 低到高 → 不排。
開著排序時下面會多一條滑桿，那是**門檻**（低於這個值的直接不顯示），
和排序方向是兩件事，可以分別用「清除門檻」單獨清掉。


---
## 9 ・ 關閉服務

做完記得關掉，不然 port 會一直被佔著。

In [ ]:
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=10)
    print("服務已關閉")
else:
    print("服務不在執行中")

---
## 課程總結

六本 notebook 走下來，你做出了一個完整的系統 ——
而且**整包程式碼沒有一行是別人先寫好給你的**。

| 章節 | 學到的東西 | 產出 |
|---|---|---|
| 00 | 統一出入口、`(data, error)`、金鑰處理 | `config.py`、`http.py` |
| 01 | 沒有文件的 API、遞迴搜尋的代價、髒資料清理 | `sources.py` |
| 02 | 金鑰安全、比對不能只信第一筆、快取與平行化 | `tmdb.py` |
| 03 | 用穩定 id 跨來源去重、pandas 分析、把流程收成函式 | `merge.py` |
| 04 | 把資料塞進 prompt（RAG 的雛形）、配額與模型下架 | `gemini.py` |
| 05 | 薄殼、同網域部署、免 build 的前端、行程管理 | `server/` 全部 |

### 貫穿全課的三個想法

1. **邏輯只留一份。** `movieapp/` 是唯一真相，notebook 和網頁服務都只是使用者。
   `merge.catalog()` 是最好的例子：第 3 章寫它，第 5 章的 `/api/movies/` 直接用。

2. **所有外部請求走同一個出入口。** 因為收攏在 `fetch_json()`，
   逾時、429、回應不是 JSON 這些狀況只要處理一次。

3. **外部服務一定會出事。** 網路會斷、API 會改版、配額會用完、模型會下架。
   把錯誤當成正常的回傳值處理，一家掛掉不要拖垮整個系統。

### 接下來可以自己試試

- 加第三家影城：在 `sources.py` 寫一個同樣形狀的函式，登記到 `SOURCES` 就好
- 改進比對：現在用字串相似度，可以加上「上映年份要接近」的條件
- 讓 AI 能回答場次時間：影城 API 其實有場次資料，只是這門課沒用到
- 把待看清單存到後端：現在只在瀏覽器裡，換台電腦就沒了